# 06. Duplicate Analysis — Identification, Deduplication & Leakage Prevention

Duplicate records inflate statistical significance, distort regression slopes, and cause disastrous **data leakage** between train and test splits.

---

### Forms of Duplication
1. **Exact Duplicates**: Rows where every single column is identical. Typically caused by ETL retries or batch re-runs.
2. **Subset / Primary Key Duplicates**: Rows sharing the same business key (`customer_id`, `timestamp`) but with minor discrepancies in secondary columns.

In [1]:
import numpy as np
import pandas as pd

# Synthesize dataset with duplicate artifacts
base_data = pd.DataFrame({
    'order_id': [101, 102, 103, 104, 105, 106, 107],
    'cust_id': ['C1', 'C2', 'C3', 'C1', 'C4', 'C5', 'C2'],
    'product': ['Laptop', 'Mouse', 'Keyboard', 'Laptop', 'Monitor', 'Desk', 'Mouse'],
    'price': [1200, 25, 75, 1200, 300, 450, 30],  # Note: C2 has price 25 and 30 (subset duplicate)
    'created_at': pd.to_datetime(['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-01', '2023-01-05', '2023-01-06', '2023-01-07'])
})

# Append an exact duplicate row
df_dups = pd.concat([base_data, base_data.iloc[[2]]], ignore_index=True)
print("Dataset with Duplicates:")
print(df_dups)

Dataset with Duplicates:
   order_id cust_id   product  price created_at
0       101      C1    Laptop   1200 2023-01-01
1       102      C2     Mouse     25 2023-01-02
2       103      C3  Keyboard     75 2023-01-03
3       104      C1    Laptop   1200 2023-01-01
4       105      C4   Monitor    300 2023-01-05
5       106      C5      Desk    450 2023-01-06
6       107      C2     Mouse     30 2023-01-07
7       103      C3  Keyboard     75 2023-01-03

## 1. Detecting Exact vs Subset Duplicates

In [1]:
# 1. Exact Duplicate Detection
exact_dups = df_dups[df_dups.duplicated(keep=False)]
print(f"Exact Duplicate Count: {df_dups.duplicated().sum()}")
print("Exact Duplicate Rows:")
print(exact_dups)

# 2. Subset Duplicate Detection on Business Key (cust_id + product)
subset_dups = df_dups[df_dups.duplicated(subset=['cust_id', 'product'], keep=False)]
print(f"\nSubset Duplicate Count (cust_id + product): {df_dups.duplicated(subset=['cust_id', 'product']).sum()}")
print("Subset Duplicate Rows:")
print(subset_dups.sort_values(by=['cust_id', 'product']))

Exact Duplicate Count: 1
Exact Duplicate Rows:
   order_id cust_id   product  price created_at
2       103      C3  Keyboard     75 2023-01-03
7       103      C3  Keyboard     75 2023-01-03

Subset Duplicate Count (cust_id + product): 3
Subset Duplicate Rows:
   order_id cust_id   product  price created_at
0       101      C1    Laptop   1200 2023-01-01
3       104      C1    Laptop   1200 2023-01-01
1       102      C2     Mouse     25 2023-01-02
6       107      C2     Mouse     30 2023-01-07
2       103      C3  Keyboard     75 2023-01-03
7       103      C3  Keyboard     75 2023-01-03

## 2. Safe Deduplication Strategy

When removing duplicates:
- `keep='first'`: Keeps the earliest chronological event.
- `keep='last'`: Keeps the most recent state update.
- Always sort by timestamp before dropping duplicates to ensure deterministic outcomes.

In [1]:
# Deterministic deduplication: sort by created_at descending and keep latest
df_deduped = (
    df_dups
    .sort_values(by='created_at', ascending=False)
    .drop_duplicates(subset=['cust_id', 'product'], keep='first')
    .sort_values(by='order_id')
    .reset_index(drop=True)
)

print("Deduplicated Clean Dataset:")
print(df_deduped)

Deduplicated Clean Dataset:
   order_id cust_id   product  price created_at
0       103      C3  Keyboard     75 2023-01-03
1       104      C1    Laptop   1200 2023-01-01
2       105      C4   Monitor    300 2023-01-05
3       106      C5      Desk    450 2023-01-06
4       107      C2     Mouse     30 2023-01-07